In [7]:
# agents/neo4j_loader_agent.py

import os
from typing import Dict, Any
from dotenv import load_dotenv
from neo4j import GraphDatabase, basic_auth

# --- Project Imports ---
from utils.models import GraphState 

In [8]:
load_dotenv()

# --- 1. Define Neo4j Connection Details ---

# Ensure these are defined in your .env file!
URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

# --- 2. Define the Agent Function (LangGraph Node) ---

def write_to_neo4j(state: GraphState) -> Dict[str, Any]:
    """
    LangGraph node function to load the extracted Knowledge Graph data into Neo4j.
    """
    print("\n--- NODE: NEO4J LOADER STARTING ---")
    
    if not state.get("graph_data_list"):
        print("⚠️ WARNING: No graph data to write. Skipping Neo4j load.")
        return {"error_message": "No data in graph_data_list."}

    # We process the data from the first document in the batch loop
    graph_data = state["graph_data_list"][0]
    nodes = graph_data["nodes"]
    relationships = graph_data["relationships"]

    if not nodes and not relationships:
        print("⚠️ WARNING: Extracted graph data is empty. Skipping Neo4j load.")
        return {"error_message": None}
        
    try:
        # Establish connection to Neo4j
        driver = GraphDatabase.driver(URI, auth=basic_auth(USERNAME, PASSWORD))
        driver.verify_connectivity()
        
        with driver.session() as session:
            
            # --- Transaction to Write Nodes ---
            node_tx = session.begin_transaction()
            node_count = 0
            for node in nodes:
                # Cypher: MERGE (n:NodeLabel {id: 'NodeID'}) SET n.name = 'NodeName'
                cypher_query = (
                    f"MERGE (n:{node['type']} {{id: $id}}) "
                    f"SET n.name = $name" 
                )
                node_tx.run(cypher_query, id=node['id'], name=node['id'])
                node_count += 1
            node_tx.commit()
            
            # --- Transaction to Write Relationships ---
            rel_tx = session.begin_transaction()
            rel_count = 0
            for rel in relationships:
                # Cypher: MATCH (s), (t) WHERE s.id = 'SourceID' AND t.id = 'TargetID' 
                # MERGE (s)-[:REL_TYPE]->(t)
                cypher_query = (
                    f"MATCH (s {{id: $source_id}}), (t {{id: $target_id}}) "
                    f"MERGE (s)-[:{rel['type']}]->(t)"
                )
                rel_tx.run(cypher_query, source_id=rel['source_id'], target_id=rel['target_id'])
                rel_count += 1
            rel_tx.commit()
        
        driver.close()
        
        print(f"✅ SUCCESS: Wrote {node_count} nodes and {rel_count} relationships to Neo4j.")
        
        # Clear the graph data list from the state, as it's now in the database
        return {
            "graph_data_list": [],
            "error_message": None, 
        }

    except Exception as e:
        print(f"❌ ERROR connecting or writing to Neo4j: {e}")
        return {
            "error_message": f"Neo4j Write Error: {e}"
        }